In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv(
    "/kaggle/input/notebooks/madhavmalhotra/creating-a-smaller-dataset-for-ciciot2023/0.1percent_8classes.csv"
)

print("="*50)
print("SHAPE:", df.shape)
print("="*50)
print("\nCOLUMNS:", df.columns.tolist())
print("="*50)
print("\nCLASS DISTRIBUTION:")
print(df.iloc[:, -1].value_counts())   
print("="*50)
print("\nMISSING VALUES:", df.isnull().sum().sum())
print("\nDATATYPES:\n", df.dtypes.value_counts())
print("="*50)
print("\nMEMORY USAGE:", df.memory_usage(deep=True).sum() / 1024**2, "MB")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/kaggle/input/notebooks/madhavmalhotra/creating-a-smaller-dataset-for-ciciot2023/0.1percent_8classes.csv"
)

print("ACTUAL LABEL COLUMN:")
print(df['label'].value_counts())
print("\nLabel dtype:", df['label'].dtype)

for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype('int8')
for col in df.select_dtypes(include='float64').columns:
    df[col] = df[col].astype('float32')
for col in df.select_dtypes(include='int64').columns:
    df[col] = df[col].astype('int32')

print(f"\nMemory after dtype fix: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")

df_sample = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(frac=0.2, random_state=42)
).reset_index(drop=True)

print(f"\nSampled shape: {df_sample.shape}")
print(f"Memory after sampling: {df_sample.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print("\nFINAL CLASS DISTRIBUTION:")
print(df_sample['label'].value_counts())

drop_cols = ['label', 'magnitude']  
feature_cols = [c for c in df_sample.columns if c not in drop_cols]
X = df_sample[feature_cols].values.astype('float32')
y_raw = df_sample['label'].values


from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f"\nFeature shape: {X.shape}")
print(f"Classes: {le.classes_}")
print(f"Encoded labels: {np.unique(y)}")

del df
import gc; gc.collect()
print("\n✅ RAM freed — ready for FL split")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

obj_cols = df_sample.select_dtypes(include='object').columns.tolist()
obj_cols = [c for c in obj_cols if c != 'label']
print("Categorical columns to encode:", obj_cols)

for col in obj_cols:
    df_sample[col] = LabelEncoder().fit_transform(
        df_sample[col].astype(str)
    )
drop_cols = ['label', 'magnitude']
feature_cols = [c for c in df_sample.columns if c not in drop_cols]
X = df_sample[feature_cols].values.astype('float32')

le = LabelEncoder()
y = le.fit_transform(df_sample['label'].values)

print(f"\n✅ Feature matrix shape: {X.shape}")
print(f"✅ Feature count: {len(feature_cols)}")
print(f"✅ Classes: {list(le.classes_)}")
print(f"✅ Class counts: {dict(zip(le.classes_, np.bincount(y)))}")

del df_sample
import gc; gc.collect()
print("\n✅ Ready for Dirichlet split!")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def dirichlet_split(X, y, n_clients=5, alpha=0.5, seed=42):
    np.random.seed(seed)
    n_classes = len(np.unique(y))
    client_indices = [[] for _ in range(n_clients)]
    
    for cls in range(n_classes):
        cls_idx = np.where(y == cls)[0]
        np.random.shuffle(cls_idx)
        proportions = np.random.dirichlet(alpha * np.ones(n_clients))
        splits = (np.cumsum(proportions) * len(cls_idx)).astype(int)
        splits = np.concatenate([[0], splits])
        for cid in range(n_clients):
            client_indices[cid].extend(
                cls_idx[splits[cid]:splits[cid+1]].tolist()
            )
    
    clients = []
    for cid in range(n_clients):
        idx = np.array(client_indices[cid])
        np.random.shuffle(idx)
        clients.append({'X': X[idx], 'y': y[idx], 'n': len(idx)})
    return clients

N_CLIENTS = 5
clients = dirichlet_split(X_train, y_train, n_clients=N_CLIENTS, alpha=0.5)

CLASS_NAMES = le.classes_
print(f"{'Client':<8} {'N':>8}  " +
      "  ".join(f"{c[:5]:>6}" for c in CLASS_NAMES))
print("-" * 80)
for i, cl in enumerate(clients):
    counts = np.bincount(cl['y'], minlength=len(CLASS_NAMES))
    pcts   = counts / cl['n'] * 100
    row    = "  ".join(f"{p:>6.1f}" for p in pcts)
    print(f"Client {i+1:<3} {cl['n']:>8}  {row}")

print("\n✅ Non-IID split done — heterogeneous distribution confirmed")

In [ ]:
import lightgbm as lgb
import shap
from sklearn.metrics import accuracy_score, f1_score

LGB_PARAMS = {
    'objective':        'multiclass',
    'num_class':        len(CLASS_NAMES),
    'boosting_type':    'gbdt',
    'n_estimators':     200,
    'learning_rate':    0.1,
    'num_leaves':       63,
    'min_child_samples': 20,
    'n_jobs':           -1,
    'random_state':     42,
    'verbosity':        -1,
}

local_models  = []
local_shap_vecs = []  
local_weights   = []   

print(f"{'Client':<8} {'N':>8} {'ValAcc':>8} {'F1-mac':>8}")
print("-" * 40)

for i, cl in enumerate(clients):
    X_tr, X_val, y_tr, y_val = train_test_split(
        cl['X'], cl['y'], test_size=0.2,
        random_state=42, stratify=cl['y']
    )

    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(20, verbose=False),
                         lgb.log_evaluation(-1)])

    shap_sample = X_val[:500]
    explainer   = shap.TreeExplainer(model)
    shap_vals   = explainer.shap_values(shap_sample)
    mean_abs_shap = np.mean(
        [np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0
    )

    local_models.append(model)
    local_shap_vecs.append(mean_abs_shap)
    local_weights.append(cl['n'])

    acc = accuracy_score(y_val, model.predict(X_val))
    f1  = f1_score(y_val, model.predict(X_val),
                   average='macro', zero_division=0)
    print(f"Client {i+1:<3} {cl['n']:>8} {acc:>8.4f} {f1:>8.4f}")

print("\n✅ All clients trained + SHAP computed")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import spearmanr
import shap, numpy as np

n_features = X_train.shape[1]  

print("Recomputing SHAP vectors (shape-safe)...")
local_shap_vecs = []

for i, (model, cl) in enumerate(zip(local_models, clients)):
    sample = cl['X'][:min(300, len(cl['X']))]
    exp    = shap.TreeExplainer(model)
    sv     = exp.shap_values(sample)
    sv_arr = np.array(sv)

    if sv_arr.ndim == 3:
        if sv_arr.shape[0] == len(sample):
            mean_abs = np.abs(sv_arr).mean(axis=(0, 2))
        else:
            mean_abs = np.abs(sv_arr).mean(axis=(0, 1))
    else:
        mean_abs = np.abs(sv_arr).mean(axis=0)

    mean_abs = mean_abs.flatten()[:n_features]
    if len(mean_abs) < n_features:
        mean_abs = np.pad(mean_abs, (0, n_features - len(mean_abs)))

    local_shap_vecs.append(mean_abs)
    print(f"  Client {i+1} SHAP shape: {mean_abs.shape} ✅")

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
total_n     = sum(local_weights)
w_norm      = np.array(local_weights) / total_n
global_shap = np.zeros(n_features)
for w, sv in zip(w_norm, local_shap_vecs):
    global_shap += w * sv

print("\n🔑 TOP 10 FEATURES — Aggregated Federated SHAP")
print("-" * 48)
top_idx = np.argsort(global_shap)[::-1][:10]
for rank, idx in enumerate(top_idx):
    print(f"  {rank+1:>2}. {feature_cols[idx]:<28} {global_shap[idx]:.6f}")

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
proba_fed    = np.zeros((len(X_test), len(CLASS_NAMES)))
for w, m in zip(w_norm, local_models):
    proba_fed += w * m.predict_proba(X_test)

y_pred_fed = np.argmax(proba_fed, axis=1)
acc_fed    = accuracy_score(y_test, y_pred_fed)
f1_fed     = f1_score(y_test, y_pred_fed, average='macro',    zero_division=0)
f1w_fed    = f1_score(y_test, y_pred_fed, average='weighted', zero_division=0)
auc_fed    = roc_auc_score(y_test, proba_fed,
                            multi_class='ovr', average='macro')

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
import lightgbm as lgb
model_c = lgb.LGBMClassifier(**LGB_PARAMS)
model_c.fit(X_train, y_train, callbacks=[lgb.log_evaluation(-1)])

y_pred_c = model_c.predict(X_test)
proba_c  = model_c.predict_proba(X_test)
acc_c    = accuracy_score(y_test, y_pred_c)
f1_c     = f1_score(y_test, y_pred_c, average='macro',    zero_division=0)
f1w_c    = f1_score(y_test, y_pred_c, average='weighted', zero_division=0)
auc_c    = roc_auc_score(y_test, proba_c,
                          multi_class='ovr', average='macro')

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
exp_c     = shap.TreeExplainer(model_c)
sv_c      = np.array(exp_c.shap_values(X_test[:300]))
if sv_c.ndim == 3:
    if sv_c.shape[0] == 300:
        central_shap = np.abs(sv_c).mean(axis=(0, 2))
    else:
        central_shap = np.abs(sv_c).mean(axis=(0, 1))
else:
    central_shap = np.abs(sv_c).mean(axis=0)
central_shap = central_shap.flatten()[:n_features]

rho, pval = spearmanr(global_shap, central_shap)

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
print("\n" + "="*55)
print(f"{'Metric':<22} {'Federated':>12} {'Centralized':>12}")
print("="*55)
print(f"{'Accuracy':<22} {acc_fed:>12.4f} {acc_c:>12.4f}")
print(f"{'F1 (macro)':<22} {f1_fed:>12.4f} {f1_c:>12.4f}")
print(f"{'F1 (weighted)':<22} {f1w_fed:>12.4f} {f1w_c:>12.4f}")
print(f"{'AUC-ROC (macro)':<22} {auc_fed:>12.4f} {auc_c:>12.4f}")
print("="*55)
print(f"\n🏆 SHAP Consistency Score ρ = {rho:.4f}  (p={pval:.2e})")
grade = "Excellent ✅" if rho>0.9 else "Good ✅" if rho>0.8 else "Moderate ⚠️"
print(f"   Ranking agreement: {grade}")
print("\n✅ FL pipeline complete — paper table ready!")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import spearmanr
import lightgbm as lgb

N_CLASSES = len(CLASS_NAMES)  

def safe_predict_proba(model, X, n_classes):
    """Missing class থাকলে 0 দিয়ে pad করো"""
    raw   = model.predict_proba(X)
    known = model.classes_          
    full  = np.zeros((len(X), n_classes))
    for col_idx, cls in enumerate(known):
        full[:, cls] = raw[:, col_idx]
    return full

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
proba_fed = np.zeros((len(X_test), N_CLASSES))
for w, m in zip(w_norm, local_models):
    proba_fed += w * safe_predict_proba(m, X_test, N_CLASSES)

y_pred_fed = np.argmax(proba_fed, axis=1)
acc_fed    = accuracy_score(y_test, y_pred_fed)
f1_fed     = f1_score(y_test, y_pred_fed, average='macro',    zero_division=0)
f1w_fed    = f1_score(y_test, y_pred_fed, average='weighted', zero_division=0)
auc_fed    = roc_auc_score(y_test, proba_fed,
                            multi_class='ovr', average='macro')

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
model_c = lgb.LGBMClassifier(**LGB_PARAMS)
model_c.fit(X_train, y_train, callbacks=[lgb.log_evaluation(-1)])

proba_c  = safe_predict_proba(model_c, X_test, N_CLASSES)
y_pred_c = model_c.predict(X_test)
acc_c    = accuracy_score(y_test, y_pred_c)
f1_c     = f1_score(y_test, y_pred_c, average='macro',    zero_division=0)
f1w_c    = f1_score(y_test, y_pred_c, average='weighted', zero_division=0)
auc_c    = roc_auc_score(y_test, proba_c,
                          multi_class='ovr', average='macro')

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
sv_c = np.array(shap.TreeExplainer(model_c).shap_values(X_test[:300]))
if sv_c.ndim == 3:
    central_shap = (np.abs(sv_c).mean(axis=(0,2))
                    if sv_c.shape[0]==300
                    else np.abs(sv_c).mean(axis=(0,1)))
else:
    central_shap = np.abs(sv_c).mean(axis=0)
central_shap = central_shap.flatten()[:n_features]

rho, pval = spearmanr(global_shap, central_shap)

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
print("="*55)
print(f"{'Metric':<22} {'Federated':>12} {'Centralized':>12}")
print("="*55)
print(f"{'Accuracy':<22} {acc_fed:>12.4f} {acc_c:>12.4f}")
print(f"{'F1 (macro)':<22} {f1_fed:>12.4f} {f1_c:>12.4f}")
print(f"{'F1 (weighted)':<22} {f1w_fed:>12.4f} {f1w_c:>12.4f}")
print(f"{'AUC-ROC (macro)':<22} {auc_fed:>12.4f} {auc_c:>12.4f}")
print("="*55)
print(f"\n🏆 SHAP Consistency Score ρ = {rho:.4f}  (p={pval:.2e})")
grade = ("Excellent ✅" if rho>0.9 else
         "Good ✅"      if rho>0.8 else "Moderate ⚠️")
print(f"   Ranking agreement: {grade}")
print("\n✅ FL pipeline complete — paper table ready!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'figure.dpi': 150,
})

metrics_labels = ['Accuracy', 'F1\n(macro)', 'F1\n(weighted)', 'AUC-ROC\n(macro)']
fed_vals  = [acc_fed,  f1_fed,  f1w_fed,  auc_fed]
cent_vals = [acc_c,    f1_c,    f1w_c,    auc_c]
print("=== VERIFY THESE NUMBERS ===")
print(f"Fed  : Acc={acc_fed:.4f} F1-mac={f1_fed:.4f} F1-w={f1w_fed:.4f} AUC={auc_fed:.4f}")
print(f"Cent : Acc={acc_c:.4f}   F1-mac={f1_c:.4f}   F1-w={f1w_c:.4f}   AUC={auc_c:.4f}")
print(f"SHAP rho = {rho:.4f}  p = {pval:.2e}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0f1117')
COLORS = {'fed':'#4FC3F7','cent':'#FF8A65','bg':'#1a1d24',
          'text':'#E0E0E0','grid':'#2a2d34','shap':'#66BB6A'}
ax1 = axes[0]
ax1.set_facecolor(COLORS['bg'])
CLASS_NAMES_LIST = list(CLASS_NAMES)
dist_matrix = np.zeros((N_CLIENTS, len(CLASS_NAMES_LIST)))
for i, cl in enumerate(clients):
    counts = np.bincount(cl['y'], minlength=len(CLASS_NAMES_LIST))
    dist_matrix[i] = counts / cl['n'] * 100

im = ax1.imshow(dist_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
ax1.set_xticks(range(len(CLASS_NAMES_LIST)))
ax1.set_xticklabels(CLASS_NAMES_LIST, rotation=30, ha='right',
                    color=COLORS['text'], fontsize=9)
ax1.set_yticks(range(N_CLIENTS))
ax1.set_yticklabels([f'Client {i+1}' for i in range(N_CLIENTS)],
                    color=COLORS['text'])
ax1.set_title('Non-IID Dirichlet Distribution (α=0.5)',
              color=COLORS['text'], fontsize=12, pad=10)
for i in range(N_CLIENTS):
    for j in range(len(CLASS_NAMES_LIST)):
        val = dist_matrix[i, j]
        ax1.text(j, i, f'{val:.1f}', ha='center', va='center',
                 fontsize=7.5,
                 color='black' if val > 40 else COLORS['text'])
cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
cbar.set_label('Class %', color=COLORS['text'])
cbar.ax.yaxis.set_tick_params(color=COLORS['text'])
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=COLORS['text'])

ax2 = axes[1]
ax2.set_facecolor(COLORS['bg'])
x = np.arange(len(metrics_labels))
w = 0.35
b1 = ax2.bar(x-w/2, fed_vals,  w, label='Federated (FedSHAP-IDS)',
             color=COLORS['fed'],  alpha=0.9, edgecolor='white', lw=0.5)
b2 = ax2.bar(x+w/2, cent_vals, w, label='Centralized Baseline',
             color=COLORS['cent'], alpha=0.9, edgecolor='white', lw=0.5)
for bar in list(b1)+list(b2):
    h = bar.get_height()
    ax2.text(bar.get_x()+bar.get_width()/2, h+0.005,
             f'{h:.4f}', ha='center', va='bottom',
             fontsize=8, color=COLORS['text'])
ax2.set_xticks(x); ax2.set_xticklabels(metrics_labels, color=COLORS['text'])
ax2.set_ylim(0.0, 1.08)
ax2.set_ylabel('Score', color=COLORS['text'])
ax2.set_title('FedSHAP-IDS vs Centralized — All Metrics',
              color=COLORS['text'], fontsize=12, pad=10)
ax2.legend(facecolor=COLORS['bg'], edgecolor=COLORS['grid'],
           labelcolor=COLORS['text'], fontsize=9)
ax2.yaxis.grid(True, color=COLORS['grid'], linestyle='--', alpha=0.5)
ax2.tick_params(colors=COLORS['text'])
ax2.spines[:].set_color(COLORS['grid'])

ax3 = axes[2]
ax3.set_facecolor(COLORS['bg'])
top_idx_arr = np.argsort(global_shap)[::-1][:10]
top_feats = [feature_cols[i] for i in top_idx_arr]
top_vals  = global_shap[top_idx_arr]
bars = ax3.barh(range(10), top_vals[::-1],
                color=COLORS['shap'], alpha=0.85,
                edgecolor='white', lw=0.4)
ax3.set_yticks(range(10))
ax3.set_yticklabels(top_feats[::-1], color=COLORS['text'], fontsize=9)
ax3.set_xlabel('Mean |SHAP Value| (Aggregated)', color=COLORS['text'])
ax3.set_title('Top-10 Global Feature Importance\n(Server-Side SHAP Aggregation)',
              color=COLORS['text'], fontsize=11, pad=10)
for bar, val in zip(bars, top_vals[::-1]):
    ax3.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=8, color=COLORS['text'])
ax3.tick_params(colors=COLORS['text'])
ax3.spines[:].set_color(COLORS['grid'])
ax3.xaxis.grid(True, color=COLORS['grid'], linestyle='--', alpha=0.5)

fig.patch.set_facecolor('#0f1117')
plt.tight_layout(pad=2.5)
plt.savefig('/kaggle/working/fig2_3_4_final.png',
            dpi=200, bbox_inches='tight',
            facecolor='#0f1117', edgecolor='none')
plt.show()
print("✅ Saved: fig2_3_4_final.png")

fig2, ax = plt.subplots(figsize=(7, 6), facecolor='#0f1117')
ax.set_facecolor('#1a1d24')
g_norm = global_shap / global_shap.max()
c_norm = central_shap / central_shap.max()
ax.scatter(c_norm, g_norm, alpha=0.7,
           color='#CE93D8', edgecolors='white', lw=0.3, s=55)
m, b = np.polyfit(c_norm, g_norm, 1)
xs = np.linspace(0, 1, 100)
ax.plot(xs, m*xs+b, color='#FF8A65',
        linestyle='--', lw=2, label=f'Spearman ρ = {rho:.4f}')
for idx in top_idx_arr[:6]:
    ax.annotate(feature_cols[idx],
                (c_norm[idx], g_norm[idx]),
                xytext=(5,3), textcoords='offset points',
                fontsize=8, color='#E0E0E0', alpha=0.9)
ax.set_xlabel('Centralized SHAP (normalized)', color='#E0E0E0')
ax.set_ylabel('Federated SHAP Aggregated (normalized)', color='#E0E0E0')
ax.set_title(f'SHAP Consistency Score: ρ={rho:.4f}  (p={pval:.1e})',
             color='#E0E0E0', pad=10)
ax.legend(facecolor='#1a1d24', edgecolor='#2a2d34', labelcolor='#E0E0E0')
ax.tick_params(colors='#E0E0E0')
for spine in ax.spines.values(): spine.set_color('#2a2d34')
ax.grid(True, color='#2a2d34', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/kaggle/working/fig5_shap_consistency_final.png',
            dpi=200, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(f"✅ Saved: fig5_shap_consistency_final.png")
print(f"\n📋 FINAL NUMBERS FOR PAPER:")
print(f"Fed  : Acc={acc_fed:.4f}  F1-mac={f1_fed:.4f}  F1-w={f1w_fed:.4f}  AUC={auc_fed:.4f}")
print(f"Cent : Acc={acc_c:.4f}  F1-mac={f1_c:.4f}  F1-w={f1w_c:.4f}  AUC={auc_c:.4f}")
print(f"SHAP : rho={rho:.4f}  p={pval:.2e}")

In [ ]:
import os

print("=== UNSW-NB15 FILES ===")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'unsw' in dirname.lower() or 'nb15' in dirname.lower():
            full = os.path.join(dirname, filename)
            size = os.path.getsize(full)/1024**2
            print(f"{size:>8.1f} MB  {full}")

In [ ]:
import os, pandas as pd

BASE = "/kaggle/input/datasets/mrwellsdavid/unsw-nb15"
print("Files found:")
all_files = {}
for f in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, f)
    size = os.path.getsize(full)/1024**2
    print(f"  {size:>8.1f} MB  {f}")
    all_files[f] = full

train_file = [v for k,v in all_files.items() 
              if 'train' in k.lower()][0]
test_file  = [v for k,v in all_files.items() 
              if 'test' in k.lower() and 'testing' in k.lower()][0]

print(f"\n✅ Train: {train_file}")
print(f"✅ Test:  {test_file}")

train = pd.read_csv(train_file)
test  = pd.read_csv(test_file)

print(f"\nTrain shape: {train.shape}")
print(f"Test  shape: {test.shape}")
print(f"\nColumns: {train.columns.tolist()}")
print(f"\nLabel distribution (train):")
label_col = 'label' if 'label' in train.columns else 'Label'
print(train[label_col].value_counts())
print(f"\nAttack categories:")
cat_col = 'attack_cat' if 'attack_cat' in train.columns else 'attack_cat'
if cat_col in train.columns:
    print(train[cat_col].value_counts())
print(f"\nMissing: {train.isnull().sum().sum()}")

In [ ]:
local_models_u    = []
local_shap_vecs_u = []
local_weights_u   = []

print(f"{'Client':<8} {'N':>8} {'ValAcc':>8} {'F1-mac':>8}")
print("-"*38)

for i, cl in enumerate(clients_u):
    cls_counts = np.bincount(cl['y'],
                             minlength=N_CLS_U)
    can_stratify = np.all(cls_counts[cls_counts > 0] >= 2)

    Xtr, Xval, ytr, yval = train_test_split(
        cl['X'], cl['y'],
        test_size=0.2,
        random_state=42,
        stratify=cl['y'] if can_stratify else None
    )

    n_cls_local = len(np.unique(ytr))
    lgb_local = dict(LGB_P)
    lgb_local['num_class'] = max(n_cls_local, 2)

    model = lgb.LGBMClassifier(**lgb_local)
    model.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              callbacks=[
                  lgb.early_stopping(20, verbose=False),
                  lgb.log_evaluation(-1)
              ])

    
    samp = Xval[:min(300, len(Xval))]
    sv   = np.array(
        shap.TreeExplainer(model).shap_values(samp))
    if sv.ndim == 3:
        mean_abs = (np.abs(sv).mean(axis=(0,2))
                    if sv.shape[0] == len(samp)
                    else np.abs(sv).mean(axis=(0,1)))
    else:
        mean_abs = np.abs(sv).mean(axis=0)

    mean_abs = mean_abs.flatten()[:n_features_u]
    if len(mean_abs) < n_features_u:
        mean_abs = np.pad(
            mean_abs, (0, n_features_u - len(mean_abs)))

    local_models_u.append(model)
    local_shap_vecs_u.append(mean_abs)
    local_weights_u.append(cl['n'])

    acc = accuracy_score(yval, model.predict(Xval))
    f1  = f1_score(yval, model.predict(Xval),
                   average='macro', zero_division=0)
    print(f"Client {i+1:<3} {cl['n']:>8,} "
          f"{acc:>8.4f} {f1:>8.4f}")

print("\n✅ All 5 clients trained!")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import spearmanr

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
total_n_u     = sum(local_weights_u)
w_norm_u      = np.array(local_weights_u) / total_n_u
global_shap_u = sum(w * sv for w, sv in
                    zip(w_norm_u, local_shap_vecs_u))

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
def safe_proba(model, X, n_cls):
    raw  = model.predict_proba(X)
    full = np.zeros((len(X), n_cls))
    for ci, cls in enumerate(model.classes_):
        full[:, cls] = raw[:, ci]
    return full

proba_fed_u = np.zeros((len(X_te_u), N_CLS_U))
for w, m in zip(w_norm_u, local_models_u):
    proba_fed_u += w * safe_proba(m, X_te_u, N_CLS_U)

y_pred_fed_u = np.argmax(proba_fed_u, axis=1)
acc_fed_u  = accuracy_score(y_te_u, y_pred_fed_u)
f1_fed_u   = f1_score(y_te_u, y_pred_fed_u,
                       average='macro',    zero_division=0)
f1w_fed_u  = f1_score(y_te_u, y_pred_fed_u,
                       average='weighted', zero_division=0)
auc_fed_u  = roc_auc_score(y_te_u, proba_fed_u,
                             multi_class='ovr', average='macro')

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
model_cu = lgb.LGBMClassifier(**LGB_P)
model_cu.fit(X_tr_u, y_tr_u,
             callbacks=[lgb.log_evaluation(-1)])
proba_cu  = safe_proba(model_cu, X_te_u, N_CLS_U)
y_pred_cu = model_cu.predict(X_te_u)
acc_cu    = accuracy_score(y_te_u, y_pred_cu)
f1_cu     = f1_score(y_te_u, y_pred_cu,
                      average='macro',    zero_division=0)
f1w_cu    = f1_score(y_te_u, y_pred_cu,
                      average='weighted', zero_division=0)
auc_cu    = roc_auc_score(y_te_u, proba_cu,
                           multi_class='ovr', average='macro')

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
sv_cu = np.array(
    shap.TreeExplainer(model_cu).shap_values(X_te_u[:300]))
if sv_cu.ndim == 3:
    c_shap_u = (np.abs(sv_cu).mean(axis=(0,2))
                if sv_cu.shape[0]==300
                else np.abs(sv_cu).mean(axis=(0,1)))
else:
    c_shap_u = np.abs(sv_cu).mean(axis=0)
c_shap_u  = c_shap_u.flatten()[:n_features_u]
rho_u, pval_u = spearmanr(global_shap_u, c_shap_u)

# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
print("="*68)
print(f"{'Metric':<22} {'── CICIoT2023 ──':^20}  {'── UNSW-NB15 ──':^20}")
print(f"{'':22} {'Fed':>9} {'Cent':>9}  {'Fed':>9} {'Cent':>9}")
print("="*68)
rows = [
    ("Accuracy",       acc_fed,  acc_c,  acc_fed_u,  acc_cu),
    ("F1 (macro)",     f1_fed,   f1_c,   f1_fed_u,   f1_cu),
    ("F1 (weighted)",  f1w_fed,  f1w_c,  f1w_fed_u,  f1w_cu),
    ("AUC-ROC (macro)",auc_fed,  auc_c,  auc_fed_u,  auc_cu),
]
for name, f1, c1, f2, c2 in rows:
    d1 = "▲" if f1 >= c1 else "▼"
    d2 = "▲" if f2 >= c2 else "▼"
    print(f"{name:<22} {f1:>9.4f} {c1:>9.4f}{d1} "
          f"{f2:>9.4f} {c2:>9.4f}{d2}")
print("="*68)
print(f"{'SHAP ρ':<22} {rho:>9.4f} {'—':>9}  "
      f"{rho_u:>9.4f} {'—':>9}")
print("="*68)

print("\n🔑 TOP 5 FEATURES — UNSW-NB15 Federated SHAP")
top_u = np.argsort(global_shap_u)[::-1][:5]
for r, idx in enumerate(top_u):
    print(f"  {r+1}. {feat_cols_u[idx]:<20} {global_shap_u[idx]:.4f}")

print("\n✅ Paper Table 2 complete!")

In [ ]:
from sklearn.metrics import classification_report

print("="*55)
print("CICIoT2023 — Per-Class Report (Federated)")
print("="*55)
print(classification_report(
    y_test, y_pred_fed,
    target_names=CLASS_NAMES,
    zero_division=0, digits=4))

print("="*55)
print("UNSW-NB15 — Per-Class Report (Federated)")
print("="*55)
print(classification_report(
    y_te_u, y_pred_fed_u,
    target_names=CLASS_NAMES_U,
    zero_division=0, digits=4))

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

K_VALUES  = [3, 5, 7, 10]
ALPHA     = 0.5   # fixed
RESULTS_K = []

print(f"{'K':>4} {'Acc_fed':>10} {'F1m_fed':>10} {'AUC_fed':>10} "
      f"{'Acc_cent':>10} {'AUC_cent':>10}")
print("-" * 60)

for K in K_VALUES:
    # Non-IID split
    clients_k = dirichlet_split(X_train, y_train,
                                n_clients=K, alpha=ALPHA)

    loc_models, loc_shap, loc_w = [], [], []
    for cl in clients_k:
        Xtr, Xval, ytr, yval = train_test_split(
            cl['X'], cl['y'], test_size=0.2,
            random_state=42,
            stratify=cl['y'] if np.all(
                np.bincount(cl['y'],
                minlength=len(CLASS_NAMES))>1) else None)

        n_cls = max(len(np.unique(ytr)), 2)
        params = dict(LGB_PARAMS); params['num_class'] = n_cls
        m = lgb.LGBMClassifier(**params)
        m.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(20, verbose=False),
                         lgb.log_evaluation(-1)])
        import shap
        samp   = cl['X'][:min(300, len(cl['X']))]
        sv     = np.array(shap.TreeExplainer(m).shap_values(samp))
        if sv.ndim == 3:
            ma = (np.abs(sv).mean(axis=(0,2))
                  if sv.shape[0]==len(samp)
                  else np.abs(sv).mean(axis=(0,1)))
        else:
            ma = np.abs(sv).mean(axis=0)
        ma = ma.flatten()[:n_features]
        if len(ma) < n_features:
            ma = np.pad(ma, (0, n_features - len(ma)))

        loc_models.append(m)
        loc_shap.append(ma)
        loc_w.append(cl['n'])

 
    total = sum(loc_w)
    w_n   = np.array(loc_w) / total
    proba = np.zeros((len(X_test), len(CLASS_NAMES)))
    for w, m in zip(w_n, loc_models):
        proba += w * safe_proba(m, X_test, len(CLASS_NAMES))

    y_pred = np.argmax(proba, axis=1)
    acc_f  = accuracy_score(y_test, y_pred)
    f1_f   = f1_score(y_test, y_pred, average='macro', zero_division=0)
    auc_f  = roc_auc_score(y_test, proba,
                            multi_class='ovr', average='macro')

  
    acc_c_k = acc_c
    auc_c_k = auc_c

    RESULTS_K.append({
        'K': K, 'acc_fed': acc_f, 'f1m_fed': f1_f,
        'auc_fed': auc_f, 'acc_cent': acc_c_k, 'auc_cent': auc_c_k
    })
    print(f"{K:>4} {acc_f:>10.4f} {f1_f:>10.4f} {auc_f:>10.4f} "
          f"{acc_c_k:>10.4f} {auc_c_k:>10.4f}")

print("\n✅ K ablation done!")

In [ ]:

ALPHA_VALUES = [0.1, 0.3, 0.5, 0.7, 1.0]
K_FIXED      = 5
RESULTS_A    = []

print(f"{'alpha':>7} {'Acc_fed':>10} {'F1m_fed':>10} "
      f"{'AUC_fed':>10} {'SCS_rho':>10}")
print("-" * 55)

from scipy.stats import spearmanr

for alpha in ALPHA_VALUES:
    clients_a = dirichlet_split(X_train, y_train,
                                n_clients=K_FIXED, alpha=alpha)

    lm, ls, lw = [], [], []
    for cl in clients_a:
        can_strat = np.all(
            np.bincount(cl['y'],
            minlength=len(CLASS_NAMES))[
            np.bincount(cl['y'],
            minlength=len(CLASS_NAMES))>0] >= 2)
        Xtr,Xval,ytr,yval = train_test_split(
            cl['X'], cl['y'], test_size=0.2,
            random_state=42,
            stratify=cl['y'] if can_strat else None)

        n_cls = max(len(np.unique(ytr)), 2)
        p2 = dict(LGB_PARAMS); p2['num_class'] = n_cls
        m  = lgb.LGBMClassifier(**p2)
        m.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(20,verbose=False),
                         lgb.log_evaluation(-1)])

        samp = cl['X'][:min(300, len(cl['X']))]
        sv   = np.array(shap.TreeExplainer(m).shap_values(samp))
        if sv.ndim == 3:
            ma = (np.abs(sv).mean(axis=(0,2))
                  if sv.shape[0]==len(samp)
                  else np.abs(sv).mean(axis=(0,1)))
        else:
            ma = np.abs(sv).mean(axis=0)
        ma = ma.flatten()[:n_features]
        if len(ma) < n_features:
            ma = np.pad(ma, (0, n_features - len(ma)))
        lm.append(m); ls.append(ma); lw.append(cl['n'])

    total_a = sum(lw)
    wn_a    = np.array(lw) / total_a


    gs = sum(w*s for w,s in zip(wn_a, ls))


    pr = np.zeros((len(X_test), len(CLASS_NAMES)))
    for w,m in zip(wn_a, lm):
        pr += w * safe_proba(m, X_test, len(CLASS_NAMES))
    yp  = np.argmax(pr, axis=1)
    af  = accuracy_score(y_test, yp)
    f1f = f1_score(y_test, yp, average='macro', zero_division=0)
    auf = roc_auc_score(y_test, pr,
                        multi_class='ovr', average='macro')

    rho_a, _ = spearmanr(gs, central_shap[:n_features])

    RESULTS_A.append({'alpha': alpha, 'acc': af,
                      'f1m': f1f, 'auc': auf, 'scs': rho_a})
    print(f"{alpha:>7.1f} {af:>10.4f} {f1f:>10.4f} "
          f"{auf:>10.4f} {rho_a:>10.4f}")

print("\n✅ α ablation done!")

In [ ]:
import sys

print("="*62)
print(f"{'Item':<35} {'Size':>12} {'Unit':>10}")
print("="*62)

raw_bytes = X_train.nbytes + y_train.nbytes
print(f"{'Raw training data (CICIoT2023)':<35} "
      f"{raw_bytes/1024**2:>12.1f} {'MB':>10}")


phi_bytes = n_features * 4   
print(f"{'SHAP vector φ_k (per client)':<35} "
      f"{phi_bytes:>12} {'bytes':>10}")


import pickle
model_bytes = len(pickle.dumps(local_models[0]))
print(f"{'LightGBM model (serialized)':<35} "
      f"{model_bytes/1024:>12.1f} {'KB':>10}")


total_client = phi_bytes + model_bytes
print(f"{'Total per client per round':<35} "
      f"{total_client/1024:>12.1f} {'KB':>10}")


total_all = total_client * 5
print(f"{'Total K=5 clients (one-shot)':<35} "
      f"{total_all/1024:>12.1f} {'KB':>10}")
print("-"*62)
print(f"{'Reduction vs raw data sharing':<35} "
      f"{raw_bytes/total_all:>12.0f} {'x smaller':>10}")
print("="*62)
print("\n✅ Communication overhead computed!")

In [ ]:
# Final ablation summary print
print("\n\n=== TABLE VIII: K ABLATION ===")
print(f"{'K':<6} {'Accuracy':<12} {'F1-Macro':<12} {'AUC-ROC':<12}")
print("-"*44)
for r in RESULTS_K:
    print(f"{r['K']:<6} {r['acc_fed']:<12.4f} "
          f"{r['f1m_fed']:<12.4f} {r['auc_fed']:<12.4f}")

print("\n\n=== TABLE IX: α ABLATION ===")
print(f"{'α':<8} {'Accuracy':<12} {'F1-Macro':<12} "
      f"{'AUC-ROC':<12} {'SCS(ρ)':<10}")
print("-"*52)
for r in RESULTS_A:
    print(f"{r['alpha']:<8.1f} {r['acc']:<12.4f} "
          f"{r['f1m']:<12.4f} {r['auc']:<12.4f} {r['scs']:<10.4f}")

In [ ]:

print("=== TABLE IX: α ABLATION (Full) ===")
print(f"{'α':<8} {'Accuracy':<12} {'F1-Macro':<12} "
      f"{'AUC-ROC':<12} {'SCS(ρ)':<10}")
print("-"*54)
for r in RESULTS_A:
    print(f"{r['alpha']:<8.1f} {r['acc']:<12.4f} "
          f"{r['f1m']:<12.4f} {r['auc']:<12.4f} "
          f"{r['scs']:<10.4f}")

print("\n\n=== COMMUNICATION OVERHEAD ===")
import pickle, sys
raw_bytes   = X_train.nbytes + y_train.nbytes
phi_bytes   = n_features * 4
model_bytes = len(pickle.dumps(local_models[0]))
total_1     = phi_bytes + model_bytes
total_5     = total_1 * 5

print(f"Raw training data    : {raw_bytes/1024**2:.1f} MB")
print(f"SHAP vector φ_k      : {phi_bytes} bytes")
print(f"LightGBM model       : {model_bytes/1024:.1f} KB")
print(f"Total per client     : {total_1/1024:.1f} KB")
print(f"Total K=5 clients    : {total_5/1024:.1f} KB")
print(f"Reduction vs raw     : {raw_bytes/total_5:.0f}x smaller")

In [ ]:
import numpy as np, lightgbm as lgb, shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import spearmanr

ALPHA_REMAINING = [0.3, 0.7, 1.0]
K_FIXED = 5

print(f"{'α':<8} {'Accuracy':<12} {'F1-Macro':<12} "
      f"{'AUC-ROC':<12} {'SCS(ρ)':<10}")
print("-"*54)
print(f"{'0.1':<8} {'0.9821':<12} {'0.5365':<12} "
      f"{'0.9937':<12} {'0.8412':<10}  [confirmed]")
print(f"{'0.5':<8} {'0.9826':<12} {'0.6452':<12} "
      f"{'0.9056':<12} {'0.8708':<10}  [main result]")

RESULTS_REMAIN = []

for alpha in ALPHA_REMAINING:
    clients_a = dirichlet_split(X_train, y_train,
                                n_clients=K_FIXED, alpha=alpha)
    lm, ls, lw = [], [], []

    for cl in clients_a:
        Xtr, Xval, ytr, yval = train_test_split(
            cl['X'], cl['y'], test_size=0.2, random_state=42,
            stratify=cl['y'] if np.all(
                np.bincount(cl['y'], minlength=len(CLASS_NAMES))
                [np.bincount(cl['y'], minlength=len(CLASS_NAMES))>0] >= 2
            ) else None)
        train_cls  = np.unique(ytr)
        val_mask   = np.isin(yval, train_cls)
        Xval_safe  = Xval[val_mask]
        yval_safe  = yval[val_mask]

        n_cls = max(len(train_cls), 2)
        p2    = dict(LGB_PARAMS); p2['num_class'] = n_cls
        m     = lgb.LGBMClassifier(**p2)

        if len(Xval_safe) > 10:
            m.fit(Xtr, ytr,
                  eval_set=[(Xval_safe, yval_safe)],
                  callbacks=[lgb.early_stopping(20, verbose=False),
                             lgb.log_evaluation(-1)])
        else:
            m.fit(Xtr, ytr, callbacks=[lgb.log_evaluation(-1)])

        samp = cl['X'][:min(300, len(cl['X']))]
        sv   = np.array(shap.TreeExplainer(m).shap_values(samp))
        if sv.ndim == 3:
            ma = (np.abs(sv).mean(axis=(0,2))
                  if sv.shape[0]==len(samp)
                  else np.abs(sv).mean(axis=(0,1)))
        else:
            ma = np.abs(sv).mean(axis=0)
        ma = ma.flatten()[:n_features]
        if len(ma) < n_features:
            ma = np.pad(ma, (0, n_features - len(ma)))
        lm.append(m); ls.append(ma); lw.append(cl['n'])

    total_a = sum(lw)
    wn_a    = np.array(lw) / total_a
    gs      = sum(w*s for w,s in zip(wn_a, ls))

    pr = np.zeros((len(X_test), len(CLASS_NAMES)))
    for w, m in zip(wn_a, lm):
        pr += w * safe_proba(m, X_test, len(CLASS_NAMES))

    yp      = np.argmax(pr, axis=1)
    af      = accuracy_score(y_test, yp)
    f1f     = f1_score(y_test, yp, average='macro', zero_division=0)
    auf     = roc_auc_score(y_test, pr,
                             multi_class='ovr', average='macro')
    rho_a,_ = spearmanr(gs, central_shap[:n_features])

    RESULTS_REMAIN.append({
        'alpha':alpha, 'acc':af, 'f1m':f1f,
        'auc':auf, 'scs':rho_a})
    print(f"{alpha:<8.1f} {af:<12.4f} {f1f:<12.4f} "
          f"{auf:<12.4f} {rho_a:<10.4f}")

print("\n✅ α ablation complete!")
print("\nCOMPLETE TABLE IX:")
all_results = [
  {'alpha':0.1,'acc':0.9821,'f1m':0.5365,'auc':0.9937,'scs':0.8412},
] + RESULTS_REMAIN[:1] + [   # 0.3
  {'alpha':0.5,'acc':0.9826,'f1m':0.6452,'auc':0.9056,'scs':0.8708},
] + RESULTS_REMAIN[1:]       # 0.7, 1.0

print(f"{'α':<8} {'Accuracy':<12} {'F1-Macro':<12} "
      f"{'AUC-ROC':<12} {'SCS(ρ)':<10}")
for r in all_results:
    print(f"{r['alpha']:<8.1f} {r['acc']:<12.4f} {r['f1m']:<12.4f} "
          f"{r['auc']:<12.4f} {r['scs']:<10.4f}")